# AutoSort một detector ba lớp

Dùng bộ dữ liệu đã hợp nhất, không tải lại Roboflow và không cần API key. Quy tắc: quả lỗi ưu tiên REJECT. Chỉ số dataset không chứng minh khả năng chạy camera/băng chuyền thật. Chọn GPU trong Runtime nếu có.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.191', 'onnx==1.23.2', 'onnxruntime==1.31.1'], check=True)


In [ ]:
from pathlib import Path
import zipfile, stat, sys, subprocess, json
from google.colab import files
uploaded = files.upload()
packages = [n for n in uploaded if n.endswith('.zip')]
if len(packages) != 1:
    raise ValueError('Upload đúng một autosort-detection-training.zip')
root = Path('/content/autosort-detection')
if root.exists():
    raise ValueError('Thư mục đã tồn tại; dùng runtime mới để không ghi đè lần train trước')
with zipfile.ZipFile(packages[0]) as archive:
    if sum(i.file_size for i in archive.infolist()) > 512 * 1024**2:
        raise ValueError('Gói quá lớn')
    for item in archive.infolist():
        target = (root / item.filename).resolve()
        if not target.is_relative_to(root.resolve()) or stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError('Đường dẫn ZIP không hợp lệ')
    root.mkdir()
    archive.extractall(root)
dataset = root / 'datasets/kumquat-detection'
manifest = json.loads((dataset/'manifest.json').read_text())
print(manifest['counts'])
print('Ảnh cách ly:', len(manifest['quarantine']))


In [ ]:
import torch
has_gpu = torch.cuda.is_available()
device = '0' if has_gpu else 'cpu'
imgsz = 640 if has_gpu else 320
epochs = 50 if has_gpu else 30
freeze = 0 if has_gpu else 10
project = root / 'runs'
subprocess.run([sys.executable, 'training/train_detection.py', '--dataset', str(dataset), '--project', str(project), '--name', 'unified-colab', '--device', device, '--imgsz', str(imgsz), '--epochs', str(epochs), '--freeze', str(freeze), '--batch', '8'], cwd=root, check=True)


In [ ]:
weights = project / 'unified-colab/weights/best.pt'
artifacts = root / 'artifacts'
subprocess.run([sys.executable, 'training/export_detection.py', '--weights', str(weights), '--dataset', str(dataset), '--destination', str(artifacts), '--imgsz', str(imgsz), '--threshold', '0.7'], cwd=root, check=True)
print(json.loads((project/'unified-colab/handoff.json').read_text()))


In [ ]:
import shutil
shutil.copy2(weights, artifacts/'best.pt')
shutil.copy2(project/'unified-colab/handoff.json', artifacts/'handoff.json')
shutil.copy2(dataset/'manifest.json', artifacts/'dataset-manifest.json')
shutil.copy2(dataset/'ATTRIBUTION.txt', artifacts/'ATTRIBUTION.txt')
for name in ['results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    path = project/'unified-colab-test'/name
    if not path.exists(): path = project/'unified-colab'/name
    if path.exists(): shutil.copy2(path, artifacts/name)
archive = shutil.make_archive('/content/autosort-detection-model-artifacts', 'zip', artifacts)
files.download(archive)


## Trước khi triển khai

Rà soát từng lớp, lỗi nhận nhầm và ảnh camera thật. Parity chỉ xác nhận cách triển khai nhất quán. ROI mặc định là toàn ảnh; nhiều mục tiêu hoặc xung đột được chuyển LOW_CONFIDENCE/REJECT. Không đặt model vào hệ thống điều khiển thiết bị chỉ dựa trên mAP nội bộ.
